In [31]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col
import scipy.stats as stats
from scipy.optimize import minimize

### Question 1

##### 1. Write a Python or R function that will take beta, X, and Y and compute the negative of the Probit log-likelihood.
##### 2. Using the Boston HMDA example from the llectures, let Y be Deny, and let X contain 1's (intercept), black, pirat, hirat, chist, hmist, phist, insurance, and selfemp. Calculate the maximum likelihood estimates for beta by minimizingg the negativ of the Probit log-likelihood calculated by your function in part 1. There is no closed-form solution for the MLE in the case of Probit and therefore numerical methods must be used to solce this minimization problem. See, for example, minimize() from scipy.optimize in Python or optim() in R. 
##### 3. Compute the predicted race differential in the denial probability. To this end, use the sample means of the other covariates to compute the predicted probability of denial for a black applicant and a non-black applicant. Report the difference in these two probabilities.

In [32]:
mydata = pd.read_csv("hmda.csv")
mydata.columns

Index(['deny', 'pirat', 'hirat', 'lvrat', 'chist', 'mhist', 'phist', 'unemp',
       'selfemp', 'insurance', 'condomin', 'black', 'single', 'hschool'],
      dtype='str')

In [40]:
binary_cols = ['deny', 'black', 'chist', 'mhist', 'phist', 'insurance', 'selfemp']
for col in binary_cols:
    if col in mydata.columns and mydata[col].dtype == object:
        mydata[col] = (mydata[col].str.lower() == 'yes').astype(int)

Yvar = mydata['deny'].to_numpy(dtype=float)

covars = ['black', 'pirat', 'hirat', 'chist', 'mhist', 'phist', 'insurance', 'selfemp']
Xvar = np.column_stack([np.ones(len(mydata)), mydata[covars].to_numpy(dtype=float)])
var_names = ['const'] + covars

print("Setup completed. Xvar shape:", Xvar.shape)

Setup completed. Xvar shape: (2380, 9)


In [34]:
def sample_neg_LL(beta, Xvar, Yvar):
    Xbeta = Xvar @ (beta)
    PhiXbeta = stats.norm.cdf(Xbeta)
    PhiXbeta = np.clip(PhiXbeta, 1e-15, 1.0 - 1e-15)
    loglike = Yvar * np.log(PhiXbeta) + (1.0 - Yvar) * np.log(1.0 - PhiXbeta)
    
    return -np.sum(loglike)

##### Above is question 1, part 1

In [35]:
beta_init = np.zeros(Xvar.shape[1])

opt_result = minimize(sample_neg_LL, beta_init, args=(Xvar, Yvar), method='BFGS')
beta_hat = opt_result.x

print("Optimization Successful:", opt_result.success)
print("\nEstimated Coefficients (MLE):")
for name, val in zip(var_names, beta_hat):
    print(f"{name:>12}: {val:.5f}")


Optimization Successful: True

Estimated Coefficients (MLE):
       const: -3.01996
       black: 0.44814
       pirat: 2.47287
       hirat: -0.12543
       chist: 0.15708
       mhist: 0.18360
       phist: 0.72083
   insurance: 2.66984
     selfemp: 0.33462


##### 2. The maximum likelihood estimate for the coefficients of black, pirat, hirat, chist, mhist, phist, insurance, and selfemp are 0.45, 2.47, -0.13, 0.16, 0.18, 0.72, 2.67, and 0.33 respectively. 

In [36]:
Xvar_means = np.mean(Xvar, axis=0)

Xvar_black = Xvar_means.copy()
Xvar_black[var_names.index('black')] = 1.0

Xvar_nonblack = Xvar_means.copy()
Xvar_nonblack[var_names.index('black')] = 0.0

prob_black = stats.norm.cdf(np.dot(Xvar_black, beta_hat))
prob_nonblack = stats.norm.cdf(np.dot(Xvar_nonblack, beta_hat))
race_diff = prob_black - prob_nonblack

print(f"Predicted P(Deny | Black = 1):     {prob_black:.4f}")
print(f"Predicted P(Deny | Non-Black = 0): {prob_nonblack:.4f}")
print(f"Predicted Race Differential:       {race_diff:.4f}")

Predicted P(Deny | Black = 1):     0.1607
Predicted P(Deny | Non-Black = 0): 0.0750
Predicted Race Differential:       0.0857


##### 3. The probable difference between a non-black candidate and a black candidate is that a black candidate is approximately 0.09 more likely to get denied. 

### Question 2
##### 1. Write a Python or R function that will take Bhat, X, and Y and compute the standard erros of the MLES. 
##### 2. Create a table that contains the MLEs and their standard errors for the Probit model estimated in Question 1. 
##### 3. Is the estimate of the coefficient on black statisticallly dignificant at the 5% level? Explain your answer.

In [37]:
def compute_probit_se(beta_hat, Xvar, Yvar):
    sign = np.where(Yvar == 1, 1.0, -1.0)[:, np.newaxis]
    Z = sign * Xvar
    
    zb = Z @ beta_hat
    
    phi_z = stats.norm.pdf(zb)
    Phi_z = np.clip(stats.norm.cdf(zb), 1e-15, 1.0 - 1e-15)
    m_z = phi_z / Phi_z
    
    h_z = m_z * (zb + m_z)
    
    H = (Xvar.T * h_z) @ Xvar
    
    cov_matrix = np.linalg.inv(H)
   
    se = np.sqrt(np.diag(cov_matrix))
    
    return se

##### above is question 2, part 1

In [38]:
se_hat = compute_probit_se(beta_hat, Xvar, Yvar)

z_stats = beta_hat / se_hat
p_values = 2.0 * (1.0 - stats.norm.cdf(np.abs(z_stats)))

results_df = pd.DataFrame({
    'Estimate (MLE)': beta_hat,
    'Std. Error': se_hat,
    'z-statistic': z_stats,
    'p-value': p_values
}, index=var_names)

display(results_df.round(4))

,Estimate (MLE),Std. Error,z-statistic,p-value
const,-3.0200,0.1963,-15.3856,0.0000
black,0.4481,0.0950,4.7178,0.0000
pirat,2.4729,0.5220,4.7375,0.0000
hirat,-0.1254,0.6428,-0.1951,0.8453
chist,0.1571,0.0213,7.3718,0.0000
mhist,0.1836,0.0718,2.5589,0.0105
phist,0.7208,0.1178,6.1190,0.0000
insurance,2.6698,0.2825,9.4502,0.0000
selfemp,0.3346,0.1098,3.0471,0.0023


##### above is the table for question 2, part 2

##### 3. The coefficient of black is statistically significant at the 5% level because it's p-value = 0, which is less than 0.05. 